# Phase 2 — Learned detector (Colab, GPU runtime)

Trains the frozen-DINOv2 + attention-pooling detector and wires it into the
fusion pipeline. All logic lives in repo modules (`training/`, `ai_image_id/detector/`);
this notebook only orchestrates. **Runtime → Change runtime type → GPU** for step 2.

In [1]:
# Setup — fresh-kernel-proof
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
!git clone -q https://github.com/Waranika/AI-image-Checkers.git
%cd /content/AI-image-Checkers
%pip install -q -e .
import sys; sys.path.insert(0, "/content/AI-image-Checkers")   # exposes training/

/content/AI-image-Checkers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 4.3 MB/s eta 0:00:00
  Building editable for ai-image-id (pyproject.toml) ... done


## Step 1 — Data: GenImage subset with matched confounds

Download a GenImage subset (e.g. the SDv1.4 generator split + ImageNet reals —
see https://github.com/GenImage-Dataset/GenImage for links). Then build
de-confounded train/val splits: identical resize + shared JPEG-Q distribution
for BOTH classes, so the head can't learn compression shortcuts.

In [3]:
# ── 1) Mount Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── after creating the Drive shortcut in the web UI ──
SHARED = "/content/drive/MyDrive/stable_diffusion_v_1_4"
!ls -la "$SHARED"          # must list imagenet_ai_0419_sdv4.zip AND .z01..z08

# Copy all parts to session disk first (FUSE random access is slow; 7z needs
# local, complete volumes). ~24 GB — expect 20–40 min. Run once per session.
!mkdir -p /content/zips
!cp -v "$SHARED"/imagenet_ai_0419_sdv4.z* "$SHARED"/imagenet_ai_0419_sdv4.zip /content/zips/

# Extract (7z auto-chains the .z01.. volumes from the .zip master)
!apt-get -qq install -y p7zip-full > /dev/null
!7z x -y -o/content/genimage /content/zips/imagenet_ai_0419_sdv4.zip | tail -3

# Verify layout, then prepare
from pathlib import Path
root = Path("/content/genimage/imagenet_ai_0419_sdv4")   # likely wrapper name — confirm:
!find /content/genimage -maxdepth 3 -type d

# ── 5) Prepare de-confounded splits ─────────────────────────────
from training.prepare_data import prepare_split

prepare_split(root/"train/nature", root/"train/ai", "/content/data/train",
              n_per_class=20_000, seed=0)
prepare_split(root/"val/nature",   root/"val/ai",   "/content/data/val",
              n_per_class=2_000,  seed=1)

Mounted at /content/drive
ls: /content/drive/MyDrive/stable_diffusion_v_1_4: No such file or directory
lrw------- 1 root root 0 Jul  8 22:58 /content/drive/MyDrive/stable_diffusion_v_1_4 -> /content/drive/.shortcut-targets-by-id/12xighYOtu-ryfYEUnNrSeZqrxT8P08Zy/stable_diffusion_v_1_4
'/content/drive/MyDrive/stable_diffusion_v_1_4/imagenet_ai_0419_sdv4.z01' -> '/content/zips/imagenet_ai_0419_sdv4.z01'
'/content/drive/MyDrive/stable_diffusion_v_1_4/imagenet_ai_0419_sdv4.z02' -> '/content/zips/imagenet_ai_0419_sdv4.z02'
'/content/drive/MyDrive/stable_diffusion_v_1_4/imagenet_ai_0419_sdv4.z03' -> '/content/zips/imagenet_ai_0419_sdv4.z03'
'/content/drive/MyDrive/stable_diffusion_v_1_4/imagenet_ai_0419_sdv4.z04' -> '/content/zips/imagenet_ai_0419_sdv4.z04'
'/content/drive/MyDrive/stable_diffusion_v_1_4/imagenet_ai_0419_sdv4.z05' -> '/content/zips/imagenet_ai_0419_sdv4.z05'
'/content/drive/MyDrive/stable_diffusion_v_1_4/imagenet_ai_0419_sdv4.z06' -> '/content/zips/imagenet_ai_0419_sdv4.z06'


FileNotFoundError: real directory does not exist: /content/genimage/imagenet_ai_0419_sdv4/train/nature

## Step 2 — Precompute frozen embeddings (GPU, one pass)

The expensive step, done once. `n_aug=2` adds two robustness-augmented variants
(JPEG 30–90 / resize / blur) per image so the head trains on transformed inputs
without re-running the backbone. ~GPU-hours depending on subset size; shards go
to Drive if you point out_dir there.

In [3]:
from training.embed import precompute

precompute("data/train/manifest.csv", "data/train", "emb/train", n_aug=2, device="cuda")
precompute("data/val/manifest.csv",   "data/val",   "emb/val",   n_aug=0, device="cuda")

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 271MB/s]


wrote 0 shards to emb/train


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 0 shards to emb/val


PosixPath('emb/val')

## Step 3 — Train the head (minutes, CPU is fine)

In [4]:
from training.train_head import train

ckpt = train("emb/train", "emb/val", out="head.pt", epochs=5, device="cuda")

ValueError: not enough values to unpack (expected 3, got 0)

## Step 4 — Calibrate (temperature scaling) and report ECE

In [ ]:
from training.calibrate_eval import calibrate

report = calibrate("head.val_logits.npz", out_json="head.calibration.json")
print(report)  # temperature, ECE before/after

## Step 5 — Cross-generator evaluation

Precompute embeddings for held-out generators (Midjourney, ADM, BigGAN, GLIDE…
from GenImage) and build the honest table: expect strong scores near your
training generator and degradation on modern commercial ones — that's the
finding that motivates fusion.

In [ ]:
import numpy as np, torch
from training.train_head import build_head, _load_shards
from training.calibrate_eval import cross_generator_table, _sigmoid
import json

state = torch.load("head.pt", map_location="cpu")
head = build_head(state["dim"]); head.load_state_dict(state["state_dict"]); head.eval()
T = json.load(open("head.calibration.json"))["temperature"]

def probs_for(emb_dir):
    ps, ys = [], []
    for patch, cls, y in _load_shards(emb_dir):
        with torch.no_grad():
            logits = head(torch.from_numpy(patch), torch.from_numpy(cls)).numpy()
        ps.append(_sigmoid(logits / T)); ys.append(y)
    return np.concatenate(ps), np.concatenate(ys).astype(np.float64)

results = {g: probs_for(f"emb/test_{g}") for g in ["midjourney", "adm", "biggan", "glide"]}
print(cross_generator_table(results))

## Step 6 — Use it in the pipeline

Point the pipeline at the checkpoint (arg or `AI_IMAGE_ID_HEAD` env var). The
fusion engine now uses the calibrated score — capped at `likely`, drift-aware,
and able to output `unlikely` for confidently-camera images.

In [ ]:
from ai_image_id.main import analyze_image

result = analyze_image("some_image.jpg", detector_ckpt="head.pt")
print(result.ai_verdict, result.confidence)
print(result.evidence.detector)
print(result.notes)